# seed-x-rm

In [1]:
import logging
from typing import List

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoConfig, MistralForCausalLM
from safetensors.torch import load_file

In [2]:
class RewardModel:
    def __init__(self, model_dir) -> None:
        config = AutoConfig.from_pretrained(model_dir)
        # config._attn_implementation = "flash_attention_2"
        self.device = torch.device('cuda')
        self.model = MistralForCausalLM(config)
        self.model.lm_head = nn.Linear(config.hidden_size, 1, bias=False)
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        state_dict = load_file(f"{model_dir}/model.safetensors")
        self.model.load_state_dict(state_dict, strict=False)
        self.model.to(dtype=torch.bfloat16)
        self.model.to(device=self.device)
        self.model.eval()
        logging.info("Load model completed.")

    @torch.no_grad()
    def score(self, prompts, chosens) -> List[float]:
        # Concat prompt and chosen, append eos_id
        input_ids_list = [self.tokenizer.encode(prompt) + self.tokenizer.encode(chosen) + [self.tokenizer.eos_token_id] for prompt, chosen in zip(prompts, chosens)]

        # Pad sequences to the maximum length
        max_length = max(len(ids) for ids in input_ids_list)
        padded_input_ids = [ids + [self.tokenizer.pad_token_id or self.tokenizer.eos_token_id] * (max_length - len(ids)) for ids in input_ids_list]

        # Forward pass
        input_ids = torch.tensor(padded_input_ids).to(device=self.device)
        logits = self.model(input_ids).logits

        # Extract logits corresponding to eos_token_id positions
        scores = []
        for i, input_ids in enumerate(input_ids_list):
            eos_position = input_ids.index(self.tokenizer.eos_token_id)
            eos_logit = logits[i, eos_position, :].squeeze().item()
            scores.append(eos_logit)

        return scores

In [ ]:
local_model_dir = "/lustre/fsw/portfolios/llmservice/users/souyang/ckpts/llm"
model_dir = f"{local_model_dir}/seed-x-rm-7b"  
model = RewardModel(model_dir)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


[1.515625, -0.3828125]


In [8]:
prompt = ["Translate the following English sentence into Chinese:\nMay the force be with you <zh>", "Translate the following English sentence into Chinese:\nMay the force be with you <zh>"]
candidate = ["愿原力与你同在","愿原力与你同在吗"]
scores = model.score(prompt, candidate)
print(scores)

[1.515625, -1.7890625]


# m-prometheus 14b

In [1]:
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.3,
    max_tokens=1024,
)

llm = LLM(model="/lustre/fsw/portfolios/llmservice/users/souyang/ckpts/llm/m-prometheus-14b", tensor_parallel_size=1, gpu_memory_utilization=0.8)

INFO 08-14 15:40:10 [__init__.py:235] Automatically detected platform cuda.
INFO 08-14 15:40:20 [config.py:1604] Using max model len 32768
INFO 08-14 15:40:22 [config.py:2434] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-14 15:40:22 [core.py:572] Waiting for init message from front-end.
INFO 08-14 15:40:22 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='/lustre/fsw/portfolios/llmservice/users/souyang/ckpts/llm/m-prometheus-14b', speculative_config=None, tokenizer='/lustre/fsw/portfolios/llmservice/users/souyang/ckpts/llm/m-prometheus-14b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=Decodi

Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]


INFO 08-14 15:40:43 [default_loader.py:262] Loading weights took 12.49 seconds
INFO 08-14 15:40:43 [gpu_model_runner.py:1892] Model loading took 27.5681 GiB and 12.921096 seconds
INFO 08-14 15:40:52 [backends.py:530] Using cache directory: /home/souyang/.cache/vllm/torch_compile_cache/9ba2632a83/rank_0_0/backbone for vLLM's torch.compile
INFO 08-14 15:40:52 [backends.py:541] Dynamo bytecode transform time: 8.20 s
INFO 08-14 15:40:55 [backends.py:194] Cache the graph for dynamic shape for later use
INFO 08-14 15:41:20 [backends.py:215] Compiling a graph for dynamic shape takes 28.02 s
INFO 08-14 15:41:33 [monitor.py:34] torch.compile takes 36.22 s in total
INFO 08-14 15:41:34 [gpu_worker.py:255] Available KV cache memory: 29.93 GiB
INFO 08-14 15:41:34 [kv_cache_utils.py:833] GPU KV cache size: 163,472 tokens
INFO 08-14 15:41:34 [kv_cache_utils.py:837] Maximum concurrency for 32,768 tokens per request: 4.99x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:04<00:00, 14.82it/s]


INFO 08-14 15:41:39 [gpu_model_runner.py:2485] Graph capturing finished in 5 secs, took 2.67 GiB
INFO 08-14 15:41:39 [core.py:193] init engine (profile, create kv cache, warmup model) took 55.72 seconds


In [3]:
TEMPLATE = """###Task Description: An instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing a evaluation criteria are given. 
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general. 
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric. 
3. The output format should look as follows: "Feedback: (write a feedback for criteria) [RESULT] (an integer number between 1 and 5)" 
4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
Translate the following text from {source_language} to {target_language}: {source}

###Response to evaluate:
{hypothesis}

###Reference Answer (Score 5):
{reference}

###Score Rubrics: [Accuracy, Fluency, Style]
Score 1: The translation contains major errors that significantly alter the meaning of the source text. It is barely comprehensible and reads like a poor machine translation. The style is completely inconsistent with the source text.
Score 2: The translation has several inaccuracies that affect the overall meaning. It is difficult to read and understand, with frequent awkward phrasings. The style only occasionally matches the source text.
Score 3: The translation is mostly accurate but has some minor errors that don't significantly alter the meaning. It is generally understandable but lacks natural flow in some parts. The style is somewhat consistent with the source text.
Score 4: The translation is accurate with only a few negligible errors. It reads naturally for the most part, with occasional minor awkwardness. The style largely matches that of the source text.
Score 5: The translation is highly accurate, conveying the full meaning of the source text. It reads as fluently as an original text in the target language. The style perfectly captures the tone and register of the source text.

###Feedback:
"""

In [5]:
prompt = TEMPLATE.format(
    source_language="English", 
    target_language="Chinese", 
    source="The quick brown fox jumps over the lazy dog.", 
    hypothesis="快速的棕色狐狸飞过了懒惰的狗。", 
    reference="敏捷的棕色狐狸跳过了那只懒狗。"
)

In [14]:
messages = [
    {"role": "user", "content": prompt}
]

output = llm.chat(messages, sampling_params)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [15]:
print(output[0].outputs[0].text)

This translation merits a score of 2 due to several significant issues:

Accuracy:
- The word "quick" is translated literally as "快速的" when "敏捷的" would be more idiomatic
- "jumps" is incorrectly translated as "飞过" (fly over) instead of "跳过" (jump over)
- The phrase "lazy dog" is translated too literally, missing the idiomatic nature of the English phrase

Fluency:
- The sentence structure is overly rigid and follows English syntax too closely
- The word order feels unnatural in Chinese, particularly with "快速的棕色狐狸"
- The connection between words is mechanical and lacks proper Chinese linguistic flow

Style:
- The translation fails to capture the playful, idiomatic nature of the original
- The choice of words is too literal and lacks the natural expression expected in Chinese
- The overall tone comes across as stiff and mechanical rather than the casual, almost poetic quality of the original

The translation demonstrates a basic understanding of the content but fails to convey it effecti

# test reward

In [24]:
from comet import download_model, load_from_checkpoint

In [27]:
model_path = download_model("Unbabel/XCOMET-XL", saving_directory='/ckpts/llm/xcomet-xl')

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
model = load_from_checkpoint(model_path)

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

Encoder model frozen.
/opt/nemo_rl_venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [30]:
data = [
    {
        "src": "Boris Johnson teeters on edge of favour with Tory MPs", 
        "mt": "Boris Johnson ist bei Tory-Abgeordneten völlig in der Gunst", 
        "ref": "Boris Johnsons Beliebtheit bei Tory-MPs steht auf der Kippe"
    }
]
model_output = model.predict(data, batch_size=8, gpus=1)
# Segment-level scores
print (model_output.scores)

# System-level score
print (model_output.system_score)

# Score explanation (error spans)
print (model_output.metadata.error_spans)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]


[0.45751047134399414]
0.45751047134399414
[[{'text': 'ist bei', 'confidence': 0.4095492959022522, 'severity': 'critical', 'start': 13, 'end': 21}, {'text': 'Abgeordnete', 'confidence': 0.2736637592315674, 'severity': 'major', 'start': 27, 'end': 38}, {'text': 'völlig in der Gunst', 'confidence': 0.5219241380691528, 'severity': 'critical', 'start': 39, 'end': 59}]]


# test data loading

In [21]:
import numpy as np
import pandas as pd

In [3]:
import pandas as pd

df = pd.read_parquet("/data/asr/yodas/npy/parakeet-tdt-0.6b-v2_robust60-1120_langid_zh_blaser2.0-qe3.0_metricx-qe4.0_simalign/en000/manifest.parquet")

In [19]:
data = df.sample(1, random_state=42).iloc[0].to_dict()

In [23]:
np.load(data['audio_npy_path'], mmap_mode='r')[420].shape

(840, 3584)

In [20]:
data

{'audio_npy_path': '/data/asr/yodas/npy/parakeet-tdt-0.6b-v2_robust60-1120_langid_zh_blaser2.0-qe3.0_metricx-qe4.0_simalign/en000/000.npy',
 'audio_npy_row': 420,
 'audio_duration': 67.2,
 'chunk_frame_size': 14,
 'segment_info': array([{'end': 3.600000000000364, 'start': 0.3200000000001637},
        {'end': 17.04000000000042, 'start': 4.2400000000002365},
        {'end': 24.96000000000049, 'start': 17.360000000000127},
        {'end': 30.720000000000255, 'start': 25.840000000000146},
        {'end': 37.20000000000027, 'start': 31.04000000000042},
        {'end': 57.2800000000002, 'start': 38.16000000000031},
        {'end': 61.92000000000007, 'start': 57.92000000000007}],
       dtype=object),
 'src_segments': array(['Henry VIII was co-written with John Fletcher.',
        "Brian Vickers suggests that Titus Andronicus was co-written with George Peel, though Jonathan Bate, the play's most recent editor for the ardent Shakespeare, believes it to be wholly the work of Shakespeare.",
    